In [1]:
import logging
import time
from datetime import datetime
import warnings
import shutil
import json
import pickle
import torch
import sys

sys.path.append("/home/liyang/BioWuYan/MethodTest/dygmamba/src")

import os
import pandas as pd
import numpy as np
import networkx as nx
import scanpy as sc
import anndata as ad

import torch.nn as nn
from tqdm import tqdm
from collections import Counter
from itertools import islice

import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

from models.DyGMamba import DyGMamba
from models.modules import MergeLayer, MergeLayerTD

from utils.load_configs import load_link_prediction_args

from utils.DataLoader import get_model_data
from utils.DataLoader import get_idx_data_loader
from utils.utils import get_neighbor_sampler, NegativeEdgeSampler
from utils.utils import get_parameter_sizes
from utils.utils import set_random_seed
from utils.utils import convert_to_gpu, create_optimizer
from utils.EarlyStopping import EarlyStopping
from utils.metrics import get_link_prediction_metrics
from models.evaluate_models_utils import evaluate_model_link_prediction
from models.inference_grn import model_link_prediction

# Configuration

In [2]:
import os
print("********************** start ********************")

start_time = time.time()  # start the time
print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Start the job")

# get arguments
args = load_link_prediction_args(is_evaluation=False)

print("**********************device********************")
print(f"Now use device is {args.device}")

data_path = "/home/liyang/BioWuYan/MethodTest/dygmamba/res/result2/"
# os.makedirs(data_path, exist_ok = True)

********************** start ********************
[2025-12-21 11:05:25] Start the job
**********************device********************
Now use device is cuda:0


# Load Data

In [8]:


feat_path = data_path + "edge_features.npy"
edge_label_path = data_path + "edge_labels.npy"

Edge_feature = np.load(feat_path, mmap_mode="r")
Edge_feature = Edge_feature.reshape(-1,1).copy()
Edge_label = np.load(edge_label_path, mmap_mode="r")
Edge_label = Edge_label.reshape(-1,1).copy()

with open(data_path + "node_feature_data.pkl", "rb") as f:
    load_data = pickle.load(f)

Node_feature = load_data['node_feature']

Node_id = pd.read_pickle(data_path + "node_id.pkl")

graph_df = pd.read_pickle(data_path + "Graph_df.pkl")
graph_df["Unnamed"] = graph_df.index
name_list = ["Unnamed", "source_node", "target_node", "time", "label", "edge_idx"]
New_Graph = graph_df[name_list].copy()
New_Graph.columns = ['Unnamed: 0', 'u', 'i', 'ts', 'label', 'idx']

# Data analysis

In [24]:
node_raw_feature_dict, edge_raw_features, full_data= get_model_data(New_Graph,
                                                        Edge_feature,
                                                        Node_feature,
                                                        feature_dim= 172)

# Result Analysis

## TF-region data

In [3]:
from data_preprocess import filter_jaspar_tf

jaspar_tf_region_file = "/home/liyang/BioWuYan/MethodTest/Data/All2/0process/jaspar_data.h5ad"

jaspar_data = ad.read_h5ad(jaspar_tf_region_file)

adata_region_tf = filter_jaspar_tf(jaspar_data)

print(adata_region_tf)


/home/liyang/BioWuYan/conda_env/dygmamba39/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")



步骤 1: 过滤 Peaks (行)
  > 找到 72563 / 72584 个 peaks 至少有 1 个 TF 结合。


/home/liyang/BioWuYan/conda_env/dygmamba39/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")



步骤 2: 过滤 TFs (列)
  > 找到 879 / 879 个 TFs 至少结合 1 个 peak。
  > 最终形状: (72563, 879)
AnnData object with n_obs × n_vars = 72563 × 879
    uns: 'description'


/home/liyang/BioWuYan/conda_env/dygmamba39/lib/python3.9/site-packages/anndata/_core/anndata.py:1756: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [4]:
import pandas as pd

coo_matrix = adata_region_tf.X.tocoo()

# 创建一个 DataFrame 来存储 TF-Peak 的连接
tf_peak_df = pd.DataFrame({
    'Peak': adata_region_tf.obs_names[coo_matrix.row],
    'TF': adata_region_tf.var_names[coo_matrix.col],
    'value': coo_matrix.data
})
tf_peak_df

## Region-gene

In [9]:
graph_df = pd.read_pickle(data_path + "Graph_df.pkl")
graph_df["Unnamed"] = graph_df.index
name_list = ["Unnamed", "source_node", "target_node", "time", "label", "edge_idx"]
New_Graph = graph_df[name_list].copy()
New_Graph.columns = ['Unnamed: 0', 'u', 'i', 'ts', 'label', 'idx']

result_graph = New_Graph.copy()

result_path = data_path + 'my_result_run{run}.npy'

predict_edge_label = np.load(result_path)

# binary_output = (predict_edge_label > 0.5).astype(int)

result_graph["predict"] = predict_edge_label
predict_grn = result_graph.copy()

mapping_series = Node_id["name"]
predict_grn['source'] = (predict_grn['u'] - 1).map(mapping_series)
predict_grn['target'] = (predict_grn['i'] - 1).map(mapping_series)
predict_grn


,Unnamed: 0,u,i,ts,label,idx,predict,source,target
0,0,1643,1,0.103799,1,1,0.054033,chr12-53370831-53371774,AAAS
1,1,1643,1,0.130393,1,2,1.000000,chr12-53370831-53371774,AAAS
2,2,1643,1,0.427715,1,3,1.000000,chr12-53370831-53371774,AAAS
3,3,1643,1,0.487691,1,4,1.000000,chr12-53370831-53371774,AAAS
4,4,1643,1,0.560851,1,5,1.000000,chr12-53370831-53371774,AAAS
...,...,...,...,...,...,...,...,...,...
1029875,1029875,5500,5500,10.258387,1,1029876,1.000000,chrX-7147230-7148785,chrX-7147230-7148785
1029876,1029876,5500,5500,10.318362,1,1029877,1.000000,chrX-7147230-7148785,chrX-7147230-7148785
1029877,1029877,5500,5500,10.318362,1,1029878,1.000000,chrX-7147230-7148785,chrX-7147230-7148785
1029878,1029878,5500,5500,10.552336,1,1029879,1.000000,chrX-7147230-7148785,chrX-7147230-7148785


## TF-gene network

In [12]:
peak_gene_df = predict_grn[['source', 'target', 'ts','predict']].rename(
    columns={'source': 'Peak', 'target': 'Gene'}
)

merged_df = pd.merge(tf_peak_df, peak_gene_df, on='Peak')
merged_df


,Peak,TF,value,Gene,ts,predict
0,chr1-629315-630015,FOSB::JUN,1,MTCO1P12,0.000000,0.064111
1,chr1-629315-630015,FOSB::JUN,1,MTCO1P12,0.103799,0.999998
2,chr1-629315-630015,FOSB::JUN,1,MTCO1P12,0.130393,1.000000
3,chr1-629315-630015,FOSB::JUN,1,MTCO1P12,0.162789,1.000000
4,chr1-629315-630015,FOSB::JUN,1,MTCO1P12,0.306248,1.000000
...,...,...,...,...,...,...
816431229,chrX-155026675-155027933,PRDM9,1,chrX-155026675-155027933,9.691967,1.000000
816431230,chrX-155026675-155027933,PRDM9,1,chrX-155026675-155027933,9.757383,1.000000
816431231,chrX-155026675-155027933,PRDM9,1,chrX-155026675-155027933,9.757383,1.000000
816431232,chrX-155026675-155027933,PRDM9,1,chrX-155026675-155027933,9.924167,1.000000


In [14]:
condition = ~(merged_df['Gene'].str.startswith('chr'))
dygmamba_tf_gene_grn = merged_df[condition].copy()

In [16]:
tf_gene_grn = merged_df.groupby(['TF', 'Gene', 'ts']).agg(
    peak_num=('Peak', 'nunique'),   # 对 Peak 列做去重计数，新列名叫 peak_num
    avg_weight=('predict', 'mean'),  # 对 weight 列做均值，新列名叫 avg_weight
    total_weight=('predict', 'sum')  # (可选) 建议顺便算个总权重
).reset_index()

# 查看结果
print(tf_gene_grn.head())

     TF  Gene        ts  peak_num  avg_weight  total_weight
0  ALX3  AAAS  0.000000         1    0.064111      0.064111
1  ALX3  AAAS  0.103799         2    0.527016      1.054032
2  ALX3  AAAS  0.130393         2    1.000000      2.000000
3  ALX3  AAAS  0.427715         2    1.000000      2.000000
4  ALX3  AAAS  0.487691         2    1.000000      2.000000


In [17]:
tf_gene_grn = merged_df.groupby(['TF', 'Gene', 'ts'])['Peak'].nunique()

# 将结果的 Series 转换为 DataFrame，并重命名计数列
tf_gene_grn = tf_gene_grn.reset_index(name='peak_num')
tf_gene_grn

,TF,Gene,ts,peak_num
0,ALX3,AAAS,0.103799,1
1,ALX3,AAAS,0.130393,2
2,ALX3,AAAS,0.427715,2
3,ALX3,AAAS,0.487691,2
4,ALX3,AAAS,0.539682,1
...,...,...,...,...
328783103,mix-a,chrX-7147230-7148785,9.991210,1
328783104,mix-a,chrX-7147230-7148785,10.113001,1
328783105,mix-a,chrX-7147230-7148785,10.258387,1
328783106,mix-a,chrX-7147230-7148785,10.318362,1


### save data

In [18]:
tf_gene_grn.to_pickle(data_path + "new_tf_gene_grn.pkl")

# Average GRN


## method 1

In [20]:
filtered_grn = tf_gene_grn.copy()
pivoted_grn = filtered_grn.pivot_table(
    index=['TF', 'Gene'],
    columns='ts',
    values='peak_num',
    fill_value=0
)

# 2. 计算所有时间列的平均值 (axis=1)
pivoted_grn['average_peak_num'] = pivoted_grn.mean(axis=1)

# 3. 提取我们关心的列，并重置索引
average_grn = pivoted_grn[['average_peak_num']].reset_index()

# 4. 按平均连接强度排序
average_grn = average_grn.sort_values(by='average_peak_num', ascending=False)

average_grn.reset_index()

average_grn.to_pickle(data_path + "average_tf_gene_grn.pkl")

ts,TF,Gene,average_peak_num
3101809,Stat5a::Stat5b,VEGFB,3.318386
3631275,ZNF16,VEGFB,3.318386
240744,CEBPD,VEGFB,3.318386
2613740,PRRX2,VEGFB,3.318386
2169250,NR2C2,VEGFB,3.318386
...,...,...,...
837372,FOSL2,chr19-16495942-16496642,0.040359
232684,CDX4,chr6-31197067-31198470,0.040359
3314413,TGIF1,chr6-31197067-31198470,0.040359
421748,E2F3,chr6-31202961-31204378,0.040359


In [29]:
average_grn = average_grn.reset_index()

average_grn.to_pickle(data_path + "average_tf_gene_grn.pkl")

In [37]:
pivoted_grn

ts                            0.0  0.103799  0.130393  0.162789  0.306248  \
TF    Gene                                                                  
ALX3  AAAS                    0.0       1.0       2.0       0.0       0.0   
      AASDHPPT                0.0       0.0       0.0       0.0       0.0   
      ABHD6                   0.0       0.0       0.0       1.0       0.0   
      ACO2                    0.0       2.0       0.0       0.0       1.0   
      ADCY3                   0.0       0.0       0.0       0.0       0.0   
...                           ...       ...       ...       ...       ...   
mix-a chrX-49190747-49191660  1.0       0.0       1.0       0.0       1.0   
      chrX-53683390-53684529  0.0       0.0       0.0       1.0       0.0   
      chrX-53714219-53717438  0.0       0.0       0.0       0.0       0.0   
      chrX-54043829-54044967  0.0       1.0       0.0       0.0       1.0   
      chrX-7147230-7148785    0.0       0.0       1.0       0.0       0.0   

ts                            0.427715  0.487691  0.539682  0.560851  \
TF    Gene                                                             
ALX3  AAAS                         2.0       2.0       1.0       2.0   
      AASDHPPT                     0.0       0.0       0.0       0.0   
      ABHD6                        0.0       1.0       0.0       0.0   
      ACO2                         0.0       3.0       1.0       1.0   
      ADCY3                        0.0       0.0       0.0       0.0   
...                                ...       ...       ...       ...   
mix-a chrX-49190747-49191660       0.0       0.0       0.0       0.0   
      chrX-53683390-53684529       1.0       0.0       0.0       0.0   
      chrX-53714219-53717438       0.0       0.0       0.0       1.0   
      chrX-54043829-54044967       1.0       0.0       0.0       0.0   
      chrX-7147230-7148785         1.0       1.0       0.0       0.0   

ts                            0.616636  ...  10.258387  10.318362  10.448101  \
TF    Gene                              ...                                    
ALX3  AAAS                         1.0  ...        2.0        1.0        1.0   
      AASDHPPT                     0.0  ...        1.0        0.0        0.0   
      ABHD6                        1.0  ...        0.0        0.0        0.0   
      ACO2                         2.0  ...        1.0        3.0        0.0   
      ADCY3                        0.0  ...        0.0        1.0        1.0   
...                                ...  ...        ...        ...        ...   
mix-a chrX-49190747-49191660       1.0  ...        1.0        0.0        0.0   
      chrX-53683390-53684529       0.0  ...        0.0        0.0        0.0   
      chrX-53714219-53717438       0.0  ...        0.0        0.0        0.0   
      chrX-54043829-54044967       0.0  ...        1.0        0.0        1.0   
      chrX-7147230-7148785         0.0  ...        1.0        1.0        0.0   

ts                            10.506163  10.552336  10.584224  10.606858  \
TF    Gene                                                                 
ALX3  AAAS                          1.0        0.0        1.0        2.0   
      AASDHPPT                      1.0        0.0        1.0        1.0   
      ABHD6                         0.0        0.0        0.0        0.0   
      ACO2                          2.0        1.0        0.0        0.0   
      ADCY3                         1.0        0.0        0.0        0.0   
...                                 ...        ...        ...        ...   
mix-a chrX-49190747-49191660        1.0        0.0        0.0        1.0   
      chrX-53683390-53684529        1.0        0.0        0.0        2.0   
      chrX-53714219-53717438        0.0        0.0        0.0        2.0   
      chrX-54043829-54044967        1.0        0.0        0.0        1.0   
      chrX-7147230-7148785          0.0        1.0        0.0        0.0   

ts                            1